# 02 — Baseline & Initial Experiments Evaluation

**Project**: Black-and-White Image Colorization with Deep Learning  

This notebook inspects our model architectures, parameter counts, training/validation learning curves, quantitative test-set metrics (PSNR, SSIM, LPIPS), and qualitative colorization results across:
- **Baseline**: Simple CNN Autoencoder (MSE loss, Adam, `lr=2e-4`, `bs=16`, 30 epochs)
- **E1**: Simple CNN Autoencoder with Learning Rate `2e-4 -> 1e-3` (single-variable ablation)
- **E2**: U-Net with ImageNet-Pretrained ResNet-18 Encoder & Skip Connections (`L1 loss`, Adam, `lr=2e-4`, `bs=16`, 30 epochs)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src.models import build_model
from src.utils import count_parameters

for name, cfg in [
    ('SimpleCNNAutoencoder (Baseline)', {'name': 'baseline_ae', 'base_channels': 64}),
    ('UNetResNet18 (Selected Method 1)', {'name': 'unet_resnet18', 'pretrained': False}),
]:
    m = build_model(cfg)
    tot, tr = count_parameters(m)
    print(f'{name:35s} | Total params: {tot:>12,} | Trainable: {tr:>12,}')

## 1. Quantitative Test-Set Results Table

In [ ]:
summary_df = pd.read_csv(ROOT / 'results' / 'metrics_summary.csv')
cols = [
    'experiment_name', 'model', 'loss_fn', 'optimizer', 'lr', 'batch_size', 'epochs',
    'best_epoch', 'total_params', 'train_time_min', 'test_psnr', 'test_ssim', 'test_lpips'
]
display(summary_df[cols])

## 2. Training & Validation Curves

In [ ]:
for title, rel_path in [
    ('Train vs. Validation Loss Curves', 'results/plots/train_val_loss_curves.png'),
    ('Validation PSNR, SSIM, and LPIPS Progression', 'results/plots/val_metrics_comparison.png'),
]:
    p = ROOT / rel_path
    if p.exists():
        plt.figure(figsize=(15, 5))
        plt.imshow(Image.open(p))
        plt.axis('off')
        plt.title(title)
        plt.show()

## 3. Qualitative Test-Set Colorization Comparison

In [ ]:
grid_path = ROOT / 'results' / 'samples' / 'comparison_grid.png'
if grid_path.exists():
    plt.figure(figsize=(16, 18))
    plt.imshow(Image.open(grid_path))
    plt.axis('off')
    plt.show()